# Chapter 9 — Multi-Armed Bandits## Thompson Sampling, UCB1, and ε-Greedy[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**No GPU required. Runtime: ~15 minutes.**Implements all three major bandit algorithms and compares their regret curves.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)print('Ready')

## 9.1 The Bandit Problem10 slot machines (arms). Each has an unknown true mean reward. Your goal: maximise total reward over 1000 pulls. The optimal strategy must balance exploration (learning which arm is best) and exploitation (pulling the best known arm).

In [ ]:
class BanditEnvironment:    def __init__(self, n_arms=10):        self.n_arms = n_arms        self.true_means = np.random.normal(0, 1, n_arms)        self.best_arm = np.argmax(self.true_means)        self.best_reward = self.true_means[self.best_arm]        def pull(self, arm):        return np.random.normal(self.true_means[arm], 1.0)env = BanditEnvironment(10)print('True arm means:', env.true_means.round(2))print(f'Best arm: {env.best_arm} with mean reward {env.best_reward:.2f}')

## 9.2 Three Algorithms

In [ ]:
def epsilon_greedy(env, n_steps=1000, epsilon=0.1):    counts = np.zeros(env.n_arms)    means  = np.zeros(env.n_arms)    rewards = []    regrets = []    cumulative_regret = 0    for t in range(n_steps):        arm = np.random.randint(env.n_arms) if np.random.random() < epsilon else np.argmax(means)        r = env.pull(arm)        counts[arm] += 1        means[arm] += (r - means[arm]) / counts[arm]        rewards.append(r)        cumulative_regret += env.best_reward - env.true_means[arm]        regrets.append(cumulative_regret)    return rewards, regretsdef ucb1(env, n_steps=1000):    counts = np.zeros(env.n_arms)    means  = np.zeros(env.n_arms)    rewards, regrets, cum_regret = [], [], 0    for t in range(n_steps):        if t < env.n_arms:            arm = t        else:            ucb = means + np.sqrt(2 * np.log(t) / counts)            arm = np.argmax(ucb)        r = env.pull(arm)        counts[arm] += 1        means[arm] += (r - means[arm]) / counts[arm]        rewards.append(r)        cum_regret += env.best_reward - env.true_means[arm]        regrets.append(cum_regret)    return rewards, regretsdef thompson_sampling(env, n_steps=1000):    alpha = np.ones(env.n_arms)    beta  = np.ones(env.n_arms)    rewards, regrets, cum_regret = [], [], 0    # Convert to Beta via win/loss (binary version for simplicity)    for t in range(n_steps):        samples = np.random.beta(alpha, beta)        arm = np.argmax(samples)        r = env.pull(arm)        alpha[arm] += max(0, r)  # positive reward = win        beta[arm]  += max(0, -r) # negative reward = loss        rewards.append(r)        cum_regret += env.best_reward - env.true_means[arm]        regrets.append(cum_regret)    return rewards, regrets# Compare across 100 runsN_RUNS, N_STEPS = 100, 1000all_regrets = {'ε-greedy(0.1)': [], 'UCB1': [], 'Thompson': []}for _ in range(N_RUNS):    env = BanditEnvironment(10)    _, r1 = epsilon_greedy(env, N_STEPS)    _, r2 = ucb1(env, N_STEPS)    _, r3 = thompson_sampling(env, N_STEPS)    all_regrets['ε-greedy(0.1)'].append(r1)    all_regrets['UCB1'].append(r2)    all_regrets['Thompson'].append(r3)fig, ax = plt.subplots(figsize=(10,5))colors = ['tomato','steelblue','seagreen']for (name, regrets_list), color in zip(all_regrets.items(), colors):    mean_regret = np.mean(regrets_list, axis=0)    std_regret = np.std(regrets_list, axis=0)    ax.plot(mean_regret, label=name, color=color, linewidth=2)    ax.fill_between(range(N_STEPS), mean_regret-std_regret, mean_regret+std_regret, alpha=0.15, color=color)ax.set_xlabel('Step'); ax.set_ylabel('Cumulative Regret')ax.set_title('Multi-Armed Bandit: Algorithm Comparison (100 runs)')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()print('Final regrets:', {k: f'{np.mean(v)[-1]:.1f}' for k,v in all_regrets.items()})

## ✅ Thompson Sampling wins because:1. **Epistemic uncertainty**: it explicitly models uncertainty and explores where it is high2. **Automatic annealing**: exploration decreases naturally as estimates become confident3. **Sublinear regret**: O(log T) vs O(√T) for ε-greedy, same as UCB1 but lower constant